# Crop Disease Classifier — Training Notebook

End-to-end workflow: data prep → training → evaluation → export.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')
print('Working dir:', os.getcwd())

## 1. Download / prepare dataset

In [ ]:
# Option A: Create a quick sample dataset (no download needed)
!python data/download_dataset.py --source sample --output ./data/plantvillage --sample-classes 10 --sample-images 100

# Option B: Download real PlantVillage dataset via Kaggle
# !python data/download_dataset.py --source kaggle --output ./data/plantvillage

# Option C: Download via HuggingFace
# !pip install datasets
# !python data/download_dataset.py --source huggingface --output ./data/plantvillage

## 2. Explore the dataset

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import random

DATA_DIR = './data/plantvillage'
classes = sorted([d.name for d in Path(DATA_DIR).iterdir() if d.is_dir()])
print(f'Classes ({len(classes)}):', classes[:10], '...')

# Show sample images
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
for ax, cls in zip(axes.flat, random.sample(classes, min(15, len(classes)))):
    imgs = list((Path(DATA_DIR) / cls).glob('*.jpg'))
    if imgs:
        ax.imshow(Image.open(random.choice(imgs)))
        ax.set_title(cls.replace('___', '\n'), fontsize=7)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Train the model

In [ ]:
# Quick training run
!python src/train.py \
    --data ./data/plantvillage \
    --arch efficientnet_b0 \
    --epochs 20 \
    --batch-size 32 \
    --lr 0.001 \
    --freeze-epochs 3 \
    --save-dir ./models

## 4. Evaluate & visualize

In [ ]:
import json
from src.evaluate import plot_training_history, plot_confusion_matrix

# Plot training curves
plot_training_history('./models/training_history.json', './models')
from IPython.display import Image as IPImage
IPImage('./models/training_curves.png')

In [ ]:
# Print test metrics
with open('./models/test_metrics.json') as f:
    metrics = json.load(f)

print(f"Test Accuracy:  {metrics['accuracy']:.4f}")
print(f"F1 (macro):     {metrics['f1_macro']:.4f}")
print(f"F1 (weighted):  {metrics['f1_weighted']:.4f}")
print(f"Precision:      {metrics['precision_macro']:.4f}")
print(f"Recall:         {metrics['recall_macro']:.4f}")

In [ ]:
# Confusion matrix
import yaml
with open('./config.yaml') as f:
    cfg = yaml.safe_load(f)

plot_confusion_matrix(metrics['confusion_matrix'], list(json.load(open('./models/test_metrics.json'))['per_class'].keys())[:-3], './models')
IPImage('./models/confusion_matrix.png')

## 5. Test inference on a single image

In [ ]:
from src.predict import CropDiseasePredictor
from pathlib import Path
import random

predictor = CropDiseasePredictor('./models/best_model.pth')

# Pick a random test image
all_images = list(Path('./data/plantvillage').rglob('*.jpg'))
test_img = random.choice(all_images)
print(f'Testing with: {test_img}')

result = predictor.predict(str(test_img), top_k=5)
print(f"\nTop prediction: {result['top_prediction']['class_name']}")
print(f"Confidence:     {result['top_prediction']['probability']:.4f}")
print("\nTop 5:")
for p in result['predictions']:
    print(f"  {p['class_name']:<50} {p['probability']*100:.2f}%")

## 6. Launch the API locally

In [ ]:
# Run in a separate terminal:
# python scripts/run_local.py
#
# Or in a notebook background thread:
import threading, uvicorn, os
os.environ['MODEL_PATH'] = './models/best_model.pth'

def run():
    uvicorn.run('app.main:app', host='0.0.0.0', port=8000, log_level='warning')

t = threading.Thread(target=run, daemon=True)
t.start()
print('API running at http://localhost:8000')

In [ ]:
# Test the API
import requests

with open(str(test_img), 'rb') as f:
    response = requests.post('http://localhost:8000/predict', files={'file': f})

print(response.json())